In [20]:
import os
import pretty_midi
import numpy as np
import torch

In [21]:
latent_dim = 256
note_dim = 128
time_steps = 32  # 32
num_keys = 24
batch_size = 64  # 64
epochs = 5000  # 5000
n_critic = 5
lr = 0.0001
beta1 = 0.5
beta2 = 0.9
gradient_penalty_weight = 10
output_dir = "C:/mtechpracticals/semester-3/gen-ai/genai-midi-generator-v3-oop/output"
data_dir = "C:/mtechpracticals/semester-3/gen-ai/genai-midi-generator-v3-oop/data"
max_files = 1000

In [22]:
def load_data():
    X, keys = [], []
    for i, file in enumerate(os.listdir(data_dir)):
        if file.endswith((".mid", ".midi")):
            file_path = os.path.join(data_dir, file)
            roll = midi_to_piano_roll(file_path)
            print(f"{file_path}:::")
            print(roll.shape)
            if roll is not None:
                X.append(roll)
                #midi_data = pretty_midi.PrettyMIDI(file_path)
                #key_idx = estimate_key(midi_data)
                #keys.append(key_idx)
            if i >= max_files - 1:
                break
    if not X:
        raise ValueError("No valid MIDI files loaded")
    X = np.array(X)
    assert X.shape[1:] == (
        128, 32), f"Loaded data shape mismatch: {X.shape}"
    return torch.tensor(X, dtype=torch.float32).reshape(-1, 128, 32), torch.tensor(keys, dtype=torch.long)

def midi_to_piano_roll(file_path, fs=8, n_notes=128, length=32):
    try:
        midi_data = pretty_midi.PrettyMIDI(file_path)
        piano_roll = midi_data.get_piano_roll(fs=fs)
        piano_roll = (piano_roll > 0).astype(np.float32)
        if piano_roll.shape[1] < length:
            pad = length - piano_roll.shape[1]
            piano_roll = np.pad(
                piano_roll, ((0, 0), (0, pad)), mode='constant')
        else:
            piano_roll = piano_roll[:, :length]
        if piano_roll.shape != (n_notes, length):
            raise ValueError(
                f"Invalid piano roll shape: {piano_roll.shape}, expected ({n_notes}, {length})")
        return piano_roll
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

def estimate_key(midi_data):
    try:
        chroma = midi_data.get_chroma(fs=8)
        chroma_sum = np.sum(chroma, axis=1)
        major_profile = np.array(
            [6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
        minor_profile = np.array(
            [6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
        scores = []
        for i in range(12):
            shifted_major = np.roll(major_profile, i)
            shifted_minor = np.roll(minor_profile, i)
            scores.append((np.correlate(chroma_sum, shifted_major)[0], i))
            scores.append(
                (np.correlate(chroma_sum, shifted_minor)[0], i + 12))
        best_score, key_idx = max(scores)
        return key_idx
    except:
        return 0  # Default to C major

In [23]:
load_data()

C:/mtechpracticals/semester-3/gen-ai/genai-midi-generator-v3-oop/data\ashover1.mid:::
(128, 32)
C:/mtechpracticals/semester-3/gen-ai/genai-midi-generator-v3-oop/data\ashover10.mid:::
(128, 32)
C:/mtechpracticals/semester-3/gen-ai/genai-midi-generator-v3-oop/data\ashover11.mid:::
(128, 32)
C:/mtechpracticals/semester-3/gen-ai/genai-midi-generator-v3-oop/data\ashover12.mid:::
(128, 32)
C:/mtechpracticals/semester-3/gen-ai/genai-midi-generator-v3-oop/data\ashover13.mid:::
(128, 32)
C:/mtechpracticals/semester-3/gen-ai/genai-midi-generator-v3-oop/data\ashover14.mid:::
(128, 32)
C:/mtechpracticals/semester-3/gen-ai/genai-midi-generator-v3-oop/data\ashover15.mid:::
(128, 32)
C:/mtechpracticals/semester-3/gen-ai/genai-midi-generator-v3-oop/data\ashover16.mid:::
(128, 32)
C:/mtechpracticals/semester-3/gen-ai/genai-midi-generator-v3-oop/data\ashover17.mid:::
(128, 32)
C:/mtechpracticals/semester-3/gen-ai/genai-midi-generator-v3-oop/data\ashover18.mid:::
(128, 32)
C:/mtechpracticals/semester-3/g

(tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],
 
         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],
 
         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],
 
         ...,
 
         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 